# 07 - Generate Beta Matrix (Leiden RegOut)

Builds gene-module and gene-program coefficient matrices from the OLS results produced by `06_fit_linear_model.ipynb`.

## Setup

In [ ]:
source("Main.R")
source("Utilities.R")

## Configuration

In [ ]:
source("paths.R")   # SCREEN_DATA, LARGE_SCREEN_DATA, REFERENCE; see data/README.md

# Written by 06_fit_linear_model, under this screen's data root.
TEXT_DIR     <- screenPath("TextFiles")
VERSION      <- "ComboScreen"
FDR_THRESH   <- 0.1

# The module definitions are the large screen's and are committed, so both
# screens read the same ones.
GENE_MODULES_CSV  <- referencePath("ME_GeneModules_leiden_11_Modules.csv")
GUIDE_MODULES_CSV <- referencePath("ME_GuideModules_leiden_6_Modules.csv")
NEW_GENE_GROUPS_CSV <- file.path(TEXT_DIR, paste0(VERSION, "_NewGeneGroups.csv"))

KO_GENES <- c("Ambra1", "Ankfy1", "Crebbp", "Ep300", "Fbxw7", "Klhl6",
              "March6", "Pparg", "Rfwd2", "Socs3", "Syvn1", "Traf2",
              "Trip12", "Wdr82")

## 1. Load coefficient and p-value matrices

In [ ]:
load_and_process <- function(prefix, version, fdr_thresh = FDR_THRESH) {
  coef_file <- file.path(TEXT_DIR, paste0("Coefs_", prefix, version, ".csv"))
  pval_file <- file.path(TEXT_DIR, paste0("Pvals_", prefix, version, ".csv"))

  coef  <- read.csv(coef_file,  row.names = 1)
  pvals <- read.csv(pval_file,  row.names = 1)

  rownames(coef)  <- gsub("GENE_|_", "", rownames(coef))
  rownames(pvals) <- gsub("GENE_|_", "", rownames(pvals))

  pvals_adj <- apply(pvals, 2, function(x) p.adjust(x, method = "fdr", n = length(x)))
  rownames(pvals_adj) <- rownames(pvals)

  coef_norm <- coef
  coef_norm[pvals_adj > fdr_thresh] <- 0

  list(coef = coef, pvals = pvals, pvals_adj = pvals_adj, coef_norm = coef_norm)
}

combo       <- load_and_process("",       VERSION)
combo_total <- load_and_process("total_", VERSION)

coef              <- combo$coef
coef_norm         <- combo$coef_norm
pvals             <- combo$pvals
pValsAllAdj       <- combo$pvals_adj

coef_total        <- combo_total$coef
coef_norm_total   <- combo_total$coef_norm
pvals_total       <- combo_total$pvals
pValsAllAdj_total <- combo_total$pvals_adj

## 2. Gene modules

In [ ]:
geneModules <- read.csv(GENE_MODULES_CSV, row.names = 1, check.names = FALSE)
geneModules <- geneModules[geneModules$GeneName %in% colnames(pValsAllAdj), ]

## 3. Guide modules and guide-pair table

In [ ]:
guideModules <- data.frame(read.csv(GUIDE_MODULES_CSV), row.names = 1)
guideModules <- guideModules[guideModules$GuideName %in% rownames(combo$coef), ]

guideGroups <- data.frame(
  Guide1 = sapply(rownames(combo$coef), function(x) strsplit(x, ":")[[1]][1]),
  Guide2 = sapply(rownames(combo$coef), function(x) strsplit(x, ":")[[1]][2])
)
guideGroups[is.na(guideGroups$Guide1), "Guide1"] <- guideGroups[is.na(guideGroups$Guide1), "Guide2"]
guideGroups[is.na(guideGroups$Guide2), "Guide2"] <- guideGroups[is.na(guideGroups$Guide2), "Guide1"]

guideGroups$guideCombo  <- rownames(guideGroups)
guideGroups$Guide1_Group <- guideModules[guideGroups$Guide1, "NewGuideGroup"]
guideGroups$Guide2_Group <- guideModules[guideGroups$Guide2, "NewGuideGroup"]
guideGroups$GuideGroupCombo <- apply(
  guideGroups[, c("Guide1_Group", "Guide2_Group")], 1,
  function(x) paste(sort(x), collapse = "_")
)

guideGroups <- guideGroups[order(guideGroups$GuideGroupCombo, guideGroups$guideCombo), ]
guideGroups$guideComboTotal <- gsub(":", "and", rownames(guideGroups))

## 4. Restrict to gene-module genes with significant effects

In [ ]:
pValsAllAdj_geneModules <- pValsAllAdj[, geneModules$GeneName]

s2 <- apply(pValsAllAdj_geneModules, 2, function(x) length(which(x < FDR_THRESH)))
selGeneModGenes <- names(s2[s2 > 1])

coef_norm_geneModules <- coef_norm[, selGeneModGenes][rownames(guideGroups), ]
coef_norm_geneModules_total <- coef_norm_total[, selGeneModGenes][guideGroups$guideComboTotal, ]
coef_geneModules <- coef[, selGeneModGenes][rownames(guideGroups), ]

## 5. Heatmap — interaction effects in gene modules

In [ ]:
options(repr.plot.width = 28, repr.plot.height = 14)

coef_tmp <- copy(coef_norm_geneModules)
coef_tmp[coef_tmp >  0.5] <-  0.5
coef_tmp[coef_tmp < -0.5] <- -0.5

annotDFcol <- geneModules[geneModules$GeneName %in% colnames(coef_tmp), ]
annotDFcol$GeneColor <- NULL; annotDFcol$GeneName <- NULL; annotDFcol$GeneGroup <- NULL

annotDFrow <- guideGroups[rownames(coef_tmp), c("Guide1", "GuideGroupCombo")]
annotDFrow$Guide1 <- NULL

pheatmap(coef_tmp,
         cluster_cols = FALSE, cluster_rows = TRUE,
         annotation_col = annotDFcol, annotation_row = annotDFrow,
         annotation_colors = annoCols,
         main = "Interaction effects in gene modules",
         clustering_method = "ward.D2", clustering_distance_rows = "euclidean",
         color = colorRampPalette(c("navy", "white", "red"))(100))

## 6. Heatmap — total effects in gene modules

In [ ]:
options(repr.plot.width = 28, repr.plot.height = 14)

coef_tmp <- copy(coef_norm_geneModules_total)
coef_tmp[coef_tmp >  0.5] <-  0.5
coef_tmp[coef_tmp < -0.5] <- -0.5

annotDFcol <- geneModules[geneModules$GeneName %in% colnames(coef_tmp), ]
annotDFcol$GeneColor <- NULL; annotDFcol$GeneName <- NULL; annotDFcol$GeneGroup <- NULL

annotDFrow <- guideGroups[, c("guideComboTotal", "GuideGroupCombo")]
rownames(annotDFrow) <- annotDFrow$guideComboTotal
annotDFrow <- annotDFrow[rownames(coef_tmp), ]
annotDFrow$guideComboTotal <- NULL

pheatmap(coef_tmp,
         cluster_cols = FALSE, cluster_rows = FALSE,
         annotation_col = annotDFcol, annotation_row = annotDFrow,
         annotation_colors = annoCols,
         main = "All total effects",
         clustering_method = "ward.D2", clustering_distance_rows = "euclidean",
         color = colorRampPalette(c("navy", "white", "red"))(100))

## 7. Discover additional significant 'other' genes

In [ ]:
other_genes <- colnames(coef)[colnames(coef) %ni% geneModules$GeneName]
pValsAllAdj_otherGenes <- pValsAllAdj[, other_genes]

s1 <- apply(
  pValsAllAdj_otherGenes[rownames(pValsAllAdj_otherGenes) %ni% KO_GENES, ],
  2,
  function(x) length(which(x < FDR_THRESH))
)

selGenes <- names(s1[s1 > 5])
length(selGenes)

## 8. Cluster selected genes and assign new gene programs (GPC_1..5)

In [ ]:
options(repr.plot.width = 13, repr.plot.height = 10)

myCor <- cor(coef[, selGenes], method = "pearson")
myCor[myCor >  0.6] <-  0.6
myCor[myCor < -0.6] <- -0.6

k <- pheatmap(myCor,
              cluster_cols = TRUE, cluster_rows = TRUE,
              fontsize_row = 3, fontsize_col = 3,
              clustering_method = "ward.D2", clustering_distance_rows = "euclidean",
              color = colorRampPalette(c("navy", "white", "darkorange"))(200))

colOrder <- colnames(coef[, selGenes])[k$tree_col$order]

In [ ]:
ss <- cutree(k$tree_col, k = 5)
ss <- ss[colnames(coef[, selGenes])[k$tree_col$order]]

newGenes <- data.frame(newGenes = names(ss), geneGroup = ss)
newGenes$NewPrograms <- "GPC_1"
newGenes[newGenes$geneGroup == 4, "NewPrograms"] <- "GPC_2"
newGenes[newGenes$geneGroup == 3, "NewPrograms"] <- "GPC_3"
newGenes[newGenes$geneGroup == 1, "NewPrograms"] <- "GPC_4"
newGenes[newGenes$geneGroup == 2, "NewPrograms"] <- "GPC_5"
rownames(newGenes) <- newGenes$newGenes
newGenes$geneGroup <- NULL

write.csv(newGenes, NEW_GENE_GROUPS_CSV)

## 9. Heatmap — interaction effects across new gene programs

In [ ]:
options(repr.plot.width = 48, repr.plot.height = 20)

coef_tmp <- copy(coef_norm[, names(ss)])[rownames(guideGroups), ]
coef_tmp[coef_tmp >  0.5] <-  0.5
coef_tmp[coef_tmp < -0.5] <- -0.5

annotDFrow <- guideGroups[rownames(coef_tmp), c("Guide1", "GuideGroupCombo")]
annotDFrow$Guide1 <- NULL

annotDFcol <- copy(newGenes); annotDFcol$newGenes <- NULL

k <- pheatmap(coef_tmp,
              cluster_cols = FALSE, cluster_rows = TRUE,
              fontsize_row = 13,
              annotation_row = annotDFrow, annotation_col = annotDFcol,
              annotation_colors = annoCols,
              clustering_method = "ward.D", clustering_distance_rows = "euclidean",
              color = colorRampPalette(c("navy", "white", "red"))(100))

rowOrder <- rownames(coef_tmp)[k$tree_row$order]

## 10. Combine module + program selections

In [ ]:
colnames(newGenes) <- c("GeneName", "NewGeneGroup")

selGenes <- c(selGeneModGenes, newGenes$GeneName)

coef_norm_selected       <- coef_norm[, selGenes][rownames(guideGroups), ]
coef_norm_selected_total <- coef_norm_total[guideGroups$guideComboTotal, selGenes]

## 11. Heatmap — combined selected genes

In [ ]:
options(repr.plot.width = 48, repr.plot.height = 20)

coef_tmp <- copy(coef_norm_selected[rownames(guideGroups), ])
coef_tmp[coef_tmp >  0.5] <-  0.5
coef_tmp[coef_tmp < -0.5] <- -0.5

annotDFrow <- guideGroups[rownames(coef_tmp), c("Guide1", "GuideGroupCombo")]
annotDFrow$Guide1 <- NULL

annotDFcol <- rbind(geneModules[, c("GeneName", "NewGeneGroup")], newGenes)
colnames(annotDFcol) <- c("GeneName", "NewGeneAllGroups")
annotDFcol$GeneName <- NULL

pheatmap(coef_tmp,
         cluster_cols = FALSE, cluster_rows = TRUE,
         fontsize_row = 13,
         annotation_row = annotDFrow, annotation_col = annotDFcol,
         annotation_colors = annoCols,
         clustering_method = "ward.D", clustering_distance_rows = "euclidean",
         color = colorRampPalette(c("navy", "white", "red"))(100))

## 12. Pairwise interaction-count matrix

In [ ]:
nInter <- apply(coef_norm_selected, 1, function(x) length(x[x != 0]))

tp <- data.frame(matrix(0, nrow = length(KO_GENES), ncol = length(KO_GENES)))
colnames(tp) <- KO_GENES
rownames(tp) <- KO_GENES

for (i in names(nInter)) {
  if (i %ni% KO_GENES) {
    g1 <- strsplit(i, ":")[[1]][1]
    g2 <- strsplit(i, ":")[[1]][2]
    tp[g1, g2] <- nInter[i]
    tp[g2, g1] <- nInter[i]
  }
}

gSO <- guideGroups[rownames(tp), ]
gSO <- gSO[order(gSO$Guide1_Group, gSO$Guide1), ]
tp <- tp[rownames(gSO), rownames(gSO)]
tp[tp == 0] <- NA

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4.5)

tmp <- copy(tp)
tmp[tmp > 150] <- 150

annotcolrow <- gSO[, c("Guide1_Group", "guideCombo")]
colnames(annotcolrow) <- c("KO_Mod", "guideCombo")
annotcolrow$guideCombo <- NULL

pheatmap(tmp,
         cluster_cols = FALSE, cluster_rows = FALSE,
         annotation_col = annotcolrow, annotation_row = annotcolrow)

## 13. Combined gene-group annotation table

In [ ]:
allGeneGroups <- rbind(
  geneModules[, c("GeneName", "NewGeneGroup")],
  newGenes[,    c("GeneName", "NewGeneGroup")]
)
colnames(allGeneGroups) <- c("GeneName", "NewGeneAllGroups")
allGeneGroups$NewGeneAllGroups <- factor(
  allGeneGroups$NewGeneAllGroups,
  levels = unique(allGeneGroups$NewGeneAllGroups)
)

## 14. Total-effect correlation heatmap (single KOs)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 6.5)

tp <- coef_norm_total[KO_GENES, selGenes]
myCor <- cor(t(tp), method = "pearson")
myCor[myCor > 0.3] <- 0.3

gSO <- guideGroups[rownames(myCor), ]
gSO <- gSO[order(gSO$Guide1_Group, gSO$Guide1), ]
myCor <- myCor[rownames(gSO), rownames(gSO)]

annotcolrow <- gSO[, c("Guide1_Group", "guideCombo")]
colnames(annotcolrow) <- c("KO_Mod", "guideCombo")
annotcolrow$guideCombo <- NULL

diag(myCor) <- NA
pheatmap(myCor,
         cluster_cols = TRUE, cluster_rows = TRUE,
         fontsize_row = 13, fontsize_col = 13,
         annotation_col = annotcolrow, annotation_row = annotcolrow,
         annotation_colors = annoCols,
         clustering_method = "ward.D2", clustering_distance_rows = "euclidean",
         color = colorRampPalette(c("navy", "white", "darkorange"))(200))

## 15. Number of significant interactions per guide combo

In [ ]:
options(repr.plot.width = 25, repr.plot.height = 12)

sgnEffects <- apply(pValsAllAdj, 1, function(x) length(x[x < FDR_THRESH]))

gGroups <- guideGroups[names(sgnEffects), ]
gGroups$numSgnEffects <- sgnEffects
gGroups <- gGroups[gGroups$guideCombo %ni% KO_GENES, ]
gGroups <- gGroups[order(gGroups$GuideGroupCombo, -gGroups$numSgnEffects), ]
gGroups$GuideGroupCombo <- factor(gGroups$GuideGroupCombo, levels = c("M3_M3", "M3_M5", "M5_M5"))
gGroups$guideCombo <- factor(gGroups$guideCombo, levels = gGroups$guideCombo)

ggplot(gGroups, aes(x = guideCombo, y = numSgnEffects, fill = GuideGroupCombo)) +
  geom_bar(stat = "identity") +
  theme_minimal() +
  ylab("Number of significant interactions") +
  xlab("Guide combo") +
  scale_fill_manual(values = c("blue", "red", "green")) +
  theme(
    axis.text.x  = element_text(angle = 90, vjust = 0.5, hjust = 1, size = 15),
    axis.text.y  = element_text(size = 25),
    axis.title   = element_text(size = 25)
  )

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 6)

ggplot(gGroups, aes(x = GuideGroupCombo, y = numSgnEffects, fill = GuideGroupCombo)) +
  geom_boxplot(outlier.colour = "red", outlier.shape = 8, outlier.size = 4) +
  scale_fill_manual(values = c("blue", "red", "green")) +
  xlab("Guide combo") +
  ylab("Number of significant interactions") +
  theme_minimal() +
  theme(
    axis.text.x  = element_text(size = 15),
    axis.text.y  = element_text(size = 25),
    axis.title   = element_text(size = 25)
  ) +
  stat_compare_means(
    comparisons = list(
      c("M3_M3", "M3_M5"),
      c("M3_M5", "M5_M5"),
      c("M3_M3", "M5_M5")
    ),
    method = "wilcox.test"
  )